In [11]:
import hashlib
import os
import platform
import uuid
# 修改为自己的学号
STUDENT_ID = 20244080230
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")


[Conda 环境] exp01
[学号] 20244080230
[机器指纹] 2d80c6ac8fb85281
[RUN_ID] 20244080230-2d80c6ac8fb85281


## 一、字符串的乐趣

**思路**

- `is_palindrome`：先统一小写、去掉空格（以及题目示例里出现的标点），再把清洗后的字符串和它的反转比较。
- `is_abecedarian`：按题目约定只忽略空格，然后逐个比较相邻字符是否满足 `前一个 <= 后一个`。
- `remove_vowels`：用 `str.join` + 生成器过滤掉 `a e i o u`，大小写都要处理。

In [12]:
# bonjour
def is_palindrome(s):
    '''判断字符串是否为回文（忽略空格、标点与大小写）。'''
    cleaned = [ch.lower() for ch in s if ch.isalnum()]
    return cleaned == cleaned[::-1]


for text in ["level", "kayak", "pop", "tacocat", "T A C O cat",
             "rats live on no evil star", "Was it a car or a cat I saw?", "hello"]:
    print(f"is_palindrome({text!r:<30}) = {is_palindrome(text)}")

is_palindrome('level'                       ) = True
is_palindrome('kayak'                       ) = True
is_palindrome('pop'                         ) = True
is_palindrome('tacocat'                     ) = True
is_palindrome('T A C O cat'                 ) = True
is_palindrome('rats live on no evil star'   ) = True
is_palindrome('Was it a car or a cat I saw?') = True
is_palindrome('hello'                       ) = False


In [13]:
def is_abecedarian(s):
    '''判断字符串的字母是否按字母顺序排列（忽略空格，允许重复）。'''
    letters = [ch for ch in s if ch != " "]
    return all(letters[i] <= letters[i + 1] for i in range(len(letters) - 1))


for word in ["adder", "beet", "abcd efgh xyz", "dog", "cat", ""]:
    print(f"is_abecedarian({word!r:<14}) = {is_abecedarian(word)}")

is_abecedarian('adder'       ) = True
is_abecedarian('beet'        ) = True
is_abecedarian('abcd efgh xyz') = True
is_abecedarian('dog'         ) = False
is_abecedarian('cat'         ) = False
is_abecedarian(''            ) = True


In [4]:
def remove_vowels(s):
    '''返回去掉所有元音 a/e/i/o/u 之后的字符串。'''
    return "".join(ch for ch in s if ch.lower() not in "aeiou")


for s in ["cat", "audio", "AEIOU", "rhythm", ""]:
    print(f"remove_vowels({s!r:<9}) = {remove_vowels(s)!r}")

remove_vowels('cat'    ) = 'ct'
remove_vowels('audio'  ) = 'd'
remove_vowels('AEIOU'  ) = ''
remove_vowels('rhythm' ) = 'rhythm'
remove_vowels(''       ) = ''


## 二、列表的乐趣

**思路**

- `list_reverse`：先用 `isinstance(lst, list)` 检查类型，不是列表就抛 `TypeError`；反转直接用切片 `lst[::-1]`。
- `is_sorted`：只允许遍历一次，所以用单层循环比较相邻元素，一旦出现"前 > 后"立刻返回 `False`。
  这里不关心 `seq` 到底是列表、元组还是字符串——只要支持索引和比较就行，这正是 Python 动态类型的体现。

In [5]:
def list_reverse(lst):
    '''返回反转后的新列表；输入不是列表时抛出 TypeError。'''
    if not isinstance(lst, list):
        raise TypeError(f"list_reverse() 需要列表，收到 {type(lst).__name__}")
    return lst[::-1]


print("list_reverse([1, 2, 3]) =", list_reverse([1, 2, 3]))
print("list_reverse(['a', 'b']) =", list_reverse(["a", "b"]))
print("list_reverse([]) =", list_reverse([]))
try:
    list_reverse((1, 2, 3))
except TypeError as exc:
    print("list_reverse((1, 2, 3)) -> TypeError:", exc)

list_reverse([1, 2, 3]) = [3, 2, 1]
list_reverse(['a', 'b']) = ['b', 'a']
list_reverse([]) = []
list_reverse((1, 2, 3)) -> TypeError: list_reverse() 需要列表，收到 tuple


In [6]:
def is_sorted(seq):
    '''判断序列是否升序；只遍历一遍。'''
    for i in range(len(seq) - 1):
        if seq[i] > seq[i + 1]:
            return False
    return True


print("is_sorted([1, 2, 2, 3]) =", is_sorted([1, 2, 2, 3]))
print("is_sorted([1, 3, 2])    =", is_sorted([1, 3, 2]))
print("is_sorted('abcde')      =", is_sorted("abcde"), "（字符串也是序列）")
print("is_sorted((5, 3))       =", is_sorted((5, 3)))
print("is_sorted(())           =", is_sorted(()))

is_sorted([1, 2, 2, 3]) = True
is_sorted([1, 3, 2])    = False
is_sorted('abcde')      = True （字符串也是序列）
is_sorted((5, 3))       = False
is_sorted(())           = True


## 三、作为向量的元组

**思路**

- 向量用元组表示：`vec_scalar_mult` 逐元素乘标量，`vec_inner_product` 逐元素乘再加总。
- 类型检查写成"必须是元组、元素必须是 int/float"，不满足就抛 `TypeError`；长度不一致抛 `ValueError`。
- 矩阵是"元组的元组"。`check_valid_mx` 依次检查：外层是元组、每行也是元组、各行等长、元素是数值。
- `mx_vec_mult` 先复用 `check_valid_mx` 做矩阵校验，再检查向量类型和列数是否匹配，最后对每行求内积。

In [7]:
def vec_scalar_mult(t, s):
    '''返回元组 t 每个元素乘以标量 s 的新元组。'''
    if not isinstance(t, tuple):
        raise TypeError(f"t 必须是元组，收到 {type(t).__name__}")
    if not isinstance(s, (int, float)):
        raise TypeError(f"s 必须是整数或浮点数，收到 {type(s).__name__}")
    for x in t:
        if not isinstance(x, (int, float)):
            raise TypeError(f"元组元素必须是整数或浮点数，收到 {type(x).__name__}")
    return tuple(x * s for x in t)


print("vec_scalar_mult((1, 2, 3), 2)   =", vec_scalar_mult((1, 2, 3), 2))
print("vec_scalar_mult((1.5, -2.0), 3) =", vec_scalar_mult((1.5, -2.0), 3))
for bad in [([1, 2], 2), ((1, 2), "2"), ((1, "a"), 2)]:
    try:
        vec_scalar_mult(*bad)
    except TypeError as exc:
        print(f"vec_scalar_mult{bad} -> TypeError: {exc}")

vec_scalar_mult((1, 2, 3), 2)   = (2, 4, 6)
vec_scalar_mult((1.5, -2.0), 3) = (4.5, -6.0)
vec_scalar_mult([1, 2], 2) -> TypeError: t 必须是元组，收到 list
vec_scalar_mult((1, 2), '2') -> TypeError: s 必须是整数或浮点数，收到 str
vec_scalar_mult((1, 'a'), 2) -> TypeError: 元组元素必须是整数或浮点数，收到 str


In [8]:
def vec_inner_product(x, y):
    '''返回两个向量（元组）的内积，结果类型为 float。'''
    if not isinstance(x, tuple) or not isinstance(y, tuple):
        raise TypeError("vec_inner_product() 的两个参数都必须是元组")
    if len(x) != len(y):
        raise ValueError(f"两个向量长度不一致：{len(x)} 与 {len(y)}")
    return float(sum(a * b for a, b in zip(x, y)))


print("vec_inner_product((1, 2, 3), (4, 5, 6)) =",
      vec_inner_product((1, 2, 3), (4, 5, 6)))
print("返回类型:", type(vec_inner_product((1, 2), (3, 4))).__name__)

try:
    vec_inner_product([1, 2], (3, 4))
except TypeError as exc:
    print("传入列表 -> TypeError:", exc)
try:
    vec_inner_product((1, 2), (3, 4, 5))
except ValueError as exc:
    print("长度不一致 -> ValueError:", exc)

vec_inner_product((1, 2, 3), (4, 5, 6)) = 32.0
返回类型: float
传入列表 -> TypeError: vec_inner_product() 的两个参数都必须是元组
长度不一致 -> ValueError: 两个向量长度不一致：2 与 3


In [9]:
def check_valid_mx(mx):
    '''检查 mx 是否为有效矩阵：元组的元组、各行等长、元素为 int/float。'''
    if not isinstance(mx, tuple) or len(mx) == 0:
        return False
    if not all(isinstance(row, tuple) for row in mx):
        return False
    n = len(mx[0])
    for row in mx:
        if len(row) != n:
            return False
        if not all(isinstance(e, (int, float)) for e in row):
            return False
    return True


for mx in [((1, 2), (3, 4)),
           ((1, 2, 3), (4, 5, 6)),
           ((1, 2), (3,)),
           ((1, 2), [3, 4]),
           ((1, "a"), (3, 4)),
           ()]:
    print(f"check_valid_mx({mx}) = {check_valid_mx(mx)}")

check_valid_mx(((1, 2), (3, 4))) = True
check_valid_mx(((1, 2, 3), (4, 5, 6))) = True
check_valid_mx(((1, 2), (3,))) = False
check_valid_mx(((1, 2), [3, 4])) = False
check_valid_mx(((1, 'a'), (3, 4))) = False
check_valid_mx(()) = False


In [10]:
def mx_vec_mult(mx, v):
    '''矩阵（元组的元组）与向量（元组）相乘，返回结果元组。'''
    if not check_valid_mx(mx):
        raise ValueError("mx 不是有效的矩阵（应为元组的元组、各行等长、元素为数值）")
    if not isinstance(v, tuple):
        raise TypeError(f"v 必须是元组，收到 {type(v).__name__}")
    if not all(isinstance(e, (int, float)) for e in v):
        raise TypeError("向量的元素必须是整数或浮点数")
    cols = len(mx[0])
    if len(v) != cols:
        raise ValueError(f"矩阵列数({cols})与向量长度({len(v)})不一致")
    return tuple(sum(row[j] * v[j] for j in range(cols)) for row in mx)


mx = ((1, 2, 3), (4, 5, 6))
print("矩阵:", mx)
print("向量:", (1, 1, 1))
print("M·v  =", mx_vec_mult(mx, (1, 1, 1)))
print("M·v  =", mx_vec_mult(mx, (2, 0, -1)))

try:
    mx_vec_mult(mx, (1, 1))
except ValueError as exc:
    print("长度不匹配 -> ValueError:", exc)
try:
    mx_vec_mult(((1, 2), (3,)), (1, 1))
except ValueError as exc:
    print("矩阵无效 -> ValueError:", exc)
try:
    mx_vec_mult(mx, [1, 1, 1])
except TypeError as exc:
    print("向量不是元组 -> TypeError:", exc)

矩阵: ((1, 2, 3), (4, 5, 6))
向量: (1, 1, 1)
M·v  = (6, 15)
M·v  = (-1, 2)
长度不匹配 -> ValueError: 矩阵列数(3)与向量长度(2)不一致
矩阵无效 -> ValueError: mx 不是有效的矩阵（应为元组的元组、各行等长、元素为数值）
向量不是元组 -> TypeError: v 必须是元组，收到 list


## 四、更多向量的乐趣

**思路**

- `Vector` 用 `dim`（维度）和 `elements`（坐标元组）保存状态，只给维度时生成零向量。
- 构造时逐项校验：维度是整数、非负、与元素个数一致、元素是数值，否则抛 `TypeError` / `ValueError`。
- 比较运算符统一先检查"是 Vector 且维度相同"，再按元组的字典序比较。
- `dot` 求内积；`__mul__` / `__rmul__` 同时支持标量左右乘和两个向量逐元素相乘。

In [11]:
class Vector:
    '''维度 + 坐标的向量。dim 和 elements 都是实例属性。'''

    def __init__(self, dim, elements=None):
        if not isinstance(dim, int) or isinstance(dim, bool):
            raise TypeError(f"维度必须是整数，收到 {type(dim).__name__}")
        if dim < 0:
            raise ValueError(f"维度不能为负数：{dim}")
        if elements is None:
            elements = (0,) * dim              # 只给维度时：默认零向量
        if not isinstance(elements, (list, tuple)):
            raise TypeError(f"元素必须是列表或元组，收到 {type(elements).__name__}")
        if len(elements) != dim:
            raise ValueError(f"维度({dim})与元素个数({len(elements)})不一致")
        if not all(isinstance(e, (int, float)) for e in elements):
            raise TypeError("元素必须是整数或浮点数")
        self.dim = dim
        self.elements = tuple(elements)

    def __repr__(self):
        return f"Vector({self.dim}, {self.elements})"

    __str__ = __repr__

    # ---------- 比较：先校验类型与维度 ----------
    def _same_dim_or_raise(self, other, what):
        if not isinstance(other, Vector):
            return NotImplemented
        if self.dim != other.dim:
            raise ValueError(f"维度不同的向量不能{what}：{self.dim} 与 {other.dim}")
        return other

    def __eq__(self, other):
        other = self._same_dim_or_raise(other, "比较")
        return NotImplemented if other is NotImplemented else self.elements == other.elements

    def __lt__(self, other):
        other = self._same_dim_or_raise(other, "比较")
        return NotImplemented if other is NotImplemented else self.elements < other.elements

    def __le__(self, other):
        other = self._same_dim_or_raise(other, "比较")
        return NotImplemented if other is NotImplemented else self.elements <= other.elements

    def __gt__(self, other):
        other = self._same_dim_or_raise(other, "比较")
        return NotImplemented if other is NotImplemented else self.elements > other.elements

    def __ge__(self, other):
        other = self._same_dim_or_raise(other, "比较")
        return NotImplemented if other is NotImplemented else self.elements >= other.elements

    # ---------- 内积 ----------
    def dot(self, other):
        if not isinstance(other, Vector):
            raise TypeError(f"dot() 需要 Vector，收到 {type(other).__name__}")
        if self.dim != other.dim:
            raise ValueError(f"维度不一致，无法求内积：{self.dim} 与 {other.dim}")
        return sum(a * b for a, b in zip(self.elements, other.elements))

    # ---------- 乘法：标量左右乘 + 向量逐元素乘 ----------
    def __mul__(self, other):
        if isinstance(other, Vector):
            if self.dim != other.dim:
                raise ValueError(f"维度不一致，无法逐元素相乘：{self.dim} 与 {other.dim}")
            return Vector(self.dim, [a * b for a, b in zip(self.elements, other.elements)])
        if isinstance(other, (int, float)):
            return Vector(self.dim, [a * other for a in self.elements])
        return NotImplemented

    def __rmul__(self, other):
        return self.__mul__(other)


print("Vector(2, (2, 3))    =", Vector(2, (2, 3)))
print("Vector(3)            =", Vector(3), "（只给维度 -> 零向量）")
print("Vector(2, [1.5, -2]) =", Vector(2, [1.5, -2]))
sample = Vector(2, (2, 3))
print("实例属性 dim =", sample.dim, "| elements =", sample.elements)

for args in [(2.0, (1, 2)), (-1, ()), (2, (1, 2, 3)), (2, "12"), (2, (1, "a"))]:
    try:
        Vector(*args)
    except (TypeError, ValueError) as exc:
        print(f"Vector{args} -> {type(exc).__name__}: {exc}")

Vector(2, (2, 3))    = Vector(2, (2, 3))
Vector(3)            = Vector(3, (0, 0, 0)) （只给维度 -> 零向量）
Vector(2, [1.5, -2]) = Vector(2, (1.5, -2))
实例属性 dim = 2 | elements = (2, 3)
Vector(2.0, (1, 2)) -> TypeError: 维度必须是整数，收到 float
Vector(-1, ()) -> ValueError: 维度不能为负数：-1
Vector(2, (1, 2, 3)) -> ValueError: 维度(2)与元素个数(3)不一致
Vector(2, '12') -> TypeError: 元素必须是列表或元组，收到 str
Vector(2, (1, 'a')) -> TypeError: 元素必须是整数或浮点数


**维度和元素应该是类属性还是实例属性？**

两者都应是**实例属性**。每个向量的维度和坐标是它自己的数据，不同向量的坐标几乎都不同；
如果定义成类属性，所有实例会共享同一份 `dim` / `elements`，改一个会影响全部，语义上是错的。
类属性适合放"所有实例共有"的东西，比如常量、计数器或默认配置。

所以我在 `__init__` 里用 `self.dim = dim`、`self.elements = tuple(elements)` 把它们绑定到实例上。

In [12]:
v = Vector(2, (2, 4))
w = Vector(2, (2, 5))
z = Vector(2, (2, 4))

print("v =", v, "| w =", w, "| z =", z)
print("v == z :", v == z)
print("v == w :", v == w)
print("v <  w :", v < w)
print("v <= z :", v <= z)
print("w >  v :", w > v)
print("w >= w :", w >= w)

try:
    v < Vector(3, (2, 4, 1))
except ValueError as exc:
    print("维度不同 -> ValueError:", exc)

v = Vector(2, (2, 4)) | w = Vector(2, (2, 5)) | z = Vector(2, (2, 4))
v == z : True
v == w : False
v <  w : True
v <= z : True
w >  v : True
w >= w : True
维度不同 -> ValueError: 维度不同的向量不能比较：2 与 3


In [13]:
print("Vector(3, (1,2,3)) · Vector(3, (4,5,6)) =",
      Vector(3, (1, 2, 3)).dot(Vector(3, (4, 5, 6))))

try:
    Vector(2, (1, 2)).dot((3, 4))
except TypeError as exc:
    print("参数不是 Vector -> TypeError:", exc)
try:
    Vector(2, (1, 2)).dot(Vector(3, (1, 2, 3)))
except ValueError as exc:
    print("维度不一致 -> ValueError:", exc)

Vector(3, (1,2,3)) · Vector(3, (4,5,6)) = 32
参数不是 Vector -> TypeError: dot() 需要 Vector，收到 tuple
维度不一致 -> ValueError: 维度不一致，无法求内积：2 与 3


In [14]:
v = Vector(2, (2, 3))
w = Vector(2, (4, 5))

print("2 * v   =", 2 * v)
print("v * 2   =", v * 2)
print("v * w   =", v * w, "（对应位置相乘）")
print("0.5 * v =", 0.5 * v)

try:
    v * Vector(3, (1, 2, 3))
except ValueError as exc:
    print("维度不一致 -> ValueError:", exc)

2 * v   = Vector(2, (4, 6))
v * 2   = Vector(2, (4, 6))
v * w   = Vector(2, (8, 15)) （对应位置相乘）
0.5 * v = Vector(2, (1.0, 1.5))
维度不一致 -> ValueError: 维度不一致，无法逐元素相乘：2 与 3
